# 04 - Efficiency benchmarks and scaling plots

    This notebook benchmarks the model variants. It reports parameters, tokens, approximate FLOPs, latency, throughput, and peak GPU memory. These measurements help satisfy the assignment's efficiency-analysis requirement.


## Imports and paths

    This cell imports the model configs and benchmark utilities.


In [ ]:
import sys
from pathlib import Path
import torch
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path: sys.path.insert(0, str(SRC))

from dit.configs import make_dit_config, RUNS
from dit.models import make_model
from dit.benchmark import benchmark_forward

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULT_DIR = PROJECT_ROOT / "results"
FIG_DIR = PROJECT_ROOT / "figures"
RESULT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)
print("Device:", device)

## Run forward-pass efficiency benchmarks

    This cell instantiates each config and measures forward latency, throughput, approximate FLOPs, and peak memory. If a config OOMs, it writes a status rather than crashing the notebook.


In [ ]:
rows = []
for run_spec in RUNS:
    cfg = make_dit_config(run_spec["model_name"], patch_size=run_spec["patch_size"])
    print("Benchmarking", cfg.run_id)
    try:
        model = make_model(cfg)
        bs = 4 if cfg.patch_size == 2 or cfg.model_name == "DiT-S" else 8
        row = benchmark_forward(model, batch_size=bs, device=device, num_warmup=5, num_iters=20)
        row["run_id"] = cfg.run_id
    except RuntimeError as exc:
        if device.type == "cuda": torch.cuda.empty_cache()
        row = {"run_id": cfg.run_id, "model_size": cfg.model_name, "patch_size": cfg.patch_size, "status": f"oom_or_runtime_error: {exc}"}
    rows.append(row)
eff = pd.DataFrame(rows)
out_csv = RESULT_DIR / "efficiency_results.csv"
eff.to_csv(out_csv, index=False)
display(eff)
print("Saved:", out_csv)


## Plot token count and latency/memory tradeoffs

    These plots visualize the cost of decreasing patch size. The expected trend is that smaller patches create more tokens, which increases compute and memory.


In [ ]:
ok = eff[eff["status"].astype(str).str.contains("ok|cpu", na=False)].copy()
if len(ok):
    plt.figure(figsize=(6,4))
    plt.scatter(ok["tokens"], ok["forward_latency_ms"], s=80)
    for _, r in ok.iterrows():
        plt.text(r["tokens"], r["forward_latency_ms"], r["run_id"])
    plt.xlabel("Latent token count")
    plt.ylabel("Forward latency (ms)")
    plt.title("Tokens vs forward latency")
    plt.grid(True, alpha=0.3)
    out = FIG_DIR / "efficiency_tokens_vs_latency.png"
    plt.savefig(out, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", out)

    if "peak_gpu_mem_mb" in ok.columns and ok["peak_gpu_mem_mb"].notna().any():
        plt.figure(figsize=(6,4))
        plt.scatter(ok["tokens"], ok["peak_gpu_mem_mb"], s=80)
        for _, r in ok.iterrows():
            plt.text(r["tokens"], r["peak_gpu_mem_mb"], r["run_id"])
        plt.xlabel("Latent token count")
        plt.ylabel("Peak GPU memory (MB)")
        plt.title("Tokens vs peak memory")
        plt.grid(True, alpha=0.3)
        out = FIG_DIR / "efficiency_tokens_vs_memory.png"
        plt.savefig(out, dpi=200, bbox_inches="tight")
        plt.show()
        print("Saved:", out)
else:
    print("No successful efficiency rows.")


## Combine quality and efficiency results


In [ ]:
metrics_path = RESULT_DIR / "metrics.csv"
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    merged = pd.merge(metrics, eff, on="run_id", how="left", suffixes=("_metric", "_eff"))
    out = RESULT_DIR / "quality_efficiency_merged.csv"
    merged.to_csv(out, index=False)
    display(merged)
    print("Saved:", out)
else:
    print("Run 03_evaluate.ipynb first to create metrics.csv.")


## Plot quality-efficiency tradeoffs

This cell uses the merged quality and efficiency table to visualize the core short-run DiT result: higher latent token count and higher model compute generally cost more, but can improve validation loss or FID.

In [ ]:
if "merged" in globals():
    usable = merged.copy()

    # Handle duplicate/suffixed column names from the merge.
    if "tokens_metric" in usable.columns:
        usable["tokens_for_plot"] = usable["tokens_metric"]
    elif "tokens" in usable.columns:
        usable["tokens_for_plot"] = usable["tokens"]
    else:
        usable["tokens_for_plot"] = float("nan")

    if "estimated_flops" in usable.columns:
        usable["flops_for_plot"] = usable["estimated_flops"]
    else:
        usable["flops_for_plot"] = float("nan")

    usable_fid = usable[usable["fid"].notna()].copy()
    usable_loss = usable[usable["val_loss"].notna()].copy()

    if len(usable_fid):
        plt.figure(figsize=(6, 4))
        plt.scatter(usable_fid["estimated_flops"], usable_fid["fid"], s=80)
        for _, r in usable_fid.iterrows():
            plt.text(r["estimated_flops"], r["fid"], r["run_id"])
        plt.xlabel("Estimated forward FLOPs")
        plt.ylabel("FID-1K")
        plt.title("Quality vs compute: FLOPs vs FID")
        plt.grid(True, alpha=0.3)
        out = FIG_DIR / "quality_flops_vs_fid.png"
        plt.savefig(out, dpi=200, bbox_inches="tight")
        plt.show()
        print("Saved:", out)
    else:
        print("No rows with FID available for FLOPs-vs-FID plot.")

    if len(usable_loss):
        plt.figure(figsize=(6, 4))
        plt.scatter(usable_loss["estimated_flops"], usable_loss["val_loss"], s=80)
        for _, r in usable_loss.iterrows():
            plt.text(r["estimated_flops"], r["val_loss"], r["run_id"])
        plt.xlabel("Estimated forward FLOPs")
        plt.ylabel("Validation denoising loss")
        plt.title("Quality vs compute: FLOPs vs validation loss")
        plt.grid(True, alpha=0.3)
        out = FIG_DIR / "quality_flops_vs_val_loss.png"
        plt.savefig(out, dpi=200, bbox_inches="tight")
        plt.show()
        print("Saved:", out)
    else:
        print("No rows with validation loss available for FLOPs-vs-loss plot.")

    if len(usable_fid) and "peak_gpu_mem_mb" in usable_fid.columns:
        plt.figure(figsize=(6, 4))
        plt.scatter(usable_fid["peak_gpu_mem_mb"], usable_fid["fid"], s=80)
        for _, r in usable_fid.iterrows():
            plt.text(r["peak_gpu_mem_mb"], r["fid"], r["run_id"])
        plt.xlabel("Peak GPU memory (MB)")
        plt.ylabel("FID-1K")
        plt.title("Quality vs memory: peak memory vs FID")
        plt.grid(True, alpha=0.3)
        out = FIG_DIR / "quality_memory_vs_fid.png"
        plt.savefig(out, dpi=200, bbox_inches="tight")
        plt.show()
        print("Saved:", out)
else:
    print("Run the merge cell first.")

## Create compact table

This cell creates a smaller table with the most important quality and efficiency fields. 

In [ ]:
if "merged" in globals():
    report_cols = [
        "run_id",
        "model",
        "patch_size_metric",
        "tokens_metric",
        "parameters",
        "estimated_flops",
        "forward_latency_ms",
        "peak_gpu_mem_mb",
        "fid",
        "val_loss",
        "status",
    ]

    available_cols = [c for c in report_cols if c in merged.columns]
    report_table = merged[available_cols].copy()

    rename_map = {
        "patch_size_metric": "patch_size",
        "tokens_metric": "tokens",
        "estimated_flops": "forward_flops",
        "forward_latency_ms": "latency_ms",
        "peak_gpu_mem_mb": "gpu_memory_mb",
    }
    report_table = report_table.rename(columns=rename_map)

    out = RESULT_DIR / "report_summary_table.csv"
    report_table.to_csv(out, index=False)
    display(report_table)
    print("Saved:", out)
else:
    print("Run the merge cell first.")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Paths
# -----------------------------
ROOT = Path(".")
RESULT_DIR = ROOT / "results"
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

MERGED_PATH = RESULT_DIR / "quality_efficiency_merged.csv"
METRICS_PATH = RESULT_DIR / "metrics.csv"
EFF_PATH = RESULT_DIR / "efficiency_results.csv"

# -----------------------------
# Load saved results or fallback to known values
# -----------------------------
def load_results():
    if MERGED_PATH.exists():
        df = pd.read_csv(MERGED_PATH)
    elif METRICS_PATH.exists() and EFF_PATH.exists():
        metrics = pd.read_csv(METRICS_PATH)
        eff = pd.read_csv(EFF_PATH)
        df = pd.merge(metrics, eff, on="run_id", how="left", suffixes=("_metric", "_eff"))
    else:
        print("Saved result files not found. Using known values from the completed run.")
        df = pd.DataFrame({
            "run_id": ["DiT-Tiny-8", "DiT-Tiny-4", "DiT-Tiny-2", "DiT-S-4"],
            "model": ["DiT-Tiny", "DiT-Tiny", "DiT-Tiny", "DiT-S"],
            "patch_size_metric": [8, 4, 2, 4],
            "tokens_metric": [16, 64, 256, 64],
            "parameters": [4259968, 4186048, 4167568, 32515648],
            "estimated_flops": [365510657.0, 1388560385.0, 2740379646.0, 5502869502.0],
            "fid": [351.8267498599507, 295.5537968738501, 336.49958911778276, 291.4853237370286],
            "val_loss": [0.3879895377159119, 0.18914494544267654, 0.21988198548555374, 0.181574914008379],
            "t_000_100_loss": [0.8856675675574769, 0.8417330436084581, 0.8645513069629669, 0.8435019714136919],
            "t_100_300_loss": [0.5125257045030593, 0.3878671634197235, 0.43254864580777225, 0.37820936024188995],
            "t_300_600_loss": [0.2985094493627548, 0.07797626696527005, 0.10247799158096313, 0.07196885317564011],
            "t_600_1000_loss": [0.26050161957740786, 0.0031855476694181563, 0.01227958543226123, 0.0029284685442689806],
        })
    return df

df = load_results()

# -----------------------------
# Normalize column names
# -----------------------------
def get_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of these columns found: {candidates}")

run_col = get_col(df, ["run_id"])
patch_col = get_col(df, ["patch_size_metric", "patch_size", "patch_size_eff"])
tokens_col = get_col(df, ["tokens_metric", "tokens", "tokens_eff"])
params_col = get_col(df, ["parameters"])
flops_col = get_col(df, ["estimated_flops"])
fid_col = get_col(df, ["fid"])
loss_col = get_col(df, ["val_loss"])

plot_df = df.copy()
plot_df["run_label"] = plot_df[run_col].astype(str)
plot_df["patch_size"] = pd.to_numeric(plot_df[patch_col], errors="coerce")
plot_df["tokens"] = pd.to_numeric(plot_df[tokens_col], errors="coerce")
plot_df["params_m"] = pd.to_numeric(plot_df[params_col], errors="coerce") / 1e6
plot_df["gflops"] = pd.to_numeric(plot_df[flops_col], errors="coerce") / 1e9
plot_df["fid"] = pd.to_numeric(plot_df[fid_col], errors="coerce")
plot_df["val_loss"] = pd.to_numeric(plot_df[loss_col], errors="coerce")

plot_df = plot_df.dropna(subset=["gflops", "fid", "val_loss", "tokens", "params_m"]).copy()
plot_df = plot_df.sort_values("gflops")

# Save cleaned plotting table
clean_path = RESULT_DIR / "extra_plot_data.csv"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
plot_df.to_csv(clean_path, index=False)
print("Saved cleaned plotting data:", clean_path)
display(plot_df[["run_label", "patch_size", "tokens", "params_m", "gflops", "fid", "val_loss"]])

# -----------------------------
# Helper functions
# -----------------------------
def pearson_corr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        return np.nan
    return np.corrcoef(x[mask], y[mask])[0, 1]

def label_points(ax, x, y, labels):
    for xi, yi, label in zip(x, y, labels):
        ax.text(xi, yi, str(label), fontsize=8)

def savefig(name):
    out = FIG_DIR / name
    plt.savefig(out, dpi=250, bbox_inches="tight")
    plt.show()
    print("Saved:", out)

# -----------------------------
# Plot 1: Paper-style FLOPs vs FID correlation
# -----------------------------
r = pearson_corr(plot_df["gflops"], plot_df["fid"])

plt.figure(figsize=(6, 4))
plt.scatter(plot_df["gflops"], plot_df["fid"], s=90)
label_points(plt.gca(), plot_df["gflops"], plot_df["fid"], plot_df["run_label"])
plt.xlabel("Estimated forward compute (GFLOPs)")
plt.ylabel("FID-1K ↓")
plt.title(f"Compute-quality trend: GFLOPs vs FID-1K\nPearson r = {r:.3f}")
plt.grid(True, alpha=0.3)
savefig("extra_gflops_vs_fid_correlation.png")

# -----------------------------
# Plot 2: FLOPs vs validation denoising loss correlation
# -----------------------------
r = pearson_corr(plot_df["gflops"], plot_df["val_loss"])

plt.figure(figsize=(6, 4))
plt.scatter(plot_df["gflops"], plot_df["val_loss"], s=90)
label_points(plt.gca(), plot_df["gflops"], plot_df["val_loss"], plot_df["run_label"])
plt.xlabel("Estimated forward compute (GFLOPs)")
plt.ylabel("Validation denoising loss ↓")
plt.title(f"Compute-quality trend: GFLOPs vs validation loss\nPearson r = {r:.3f}")
plt.grid(True, alpha=0.3)
savefig("extra_gflops_vs_val_loss_correlation.png")

# -----------------------------
# Plot 3: Parameters vs FID
# This highlights that Tiny/8, Tiny/4, Tiny/2 have nearly fixed params but different FID.
# -----------------------------
plt.figure(figsize=(6, 4))
plt.scatter(plot_df["params_m"], plot_df["fid"], s=90)
label_points(plt.gca(), plot_df["params_m"], plot_df["fid"], plot_df["run_label"])
plt.xlabel("Parameters (millions)")
plt.ylabel("FID-1K ↓")
plt.title("Parameter count alone does not explain quality")
plt.grid(True, alpha=0.3)
savefig("extra_params_vs_fid.png")

# -----------------------------
# Plot 4: Bubble plot similar in spirit to the DiT paper
# x-axis: GFLOPs, y-axis: FID, bubble size: parameter count
# -----------------------------
bubble_sizes = 80 + 350 * (plot_df["params_m"] / plot_df["params_m"].max())

plt.figure(figsize=(6, 4))
plt.scatter(plot_df["gflops"], plot_df["fid"], s=bubble_sizes, alpha=0.7)
label_points(plt.gca(), plot_df["gflops"], plot_df["fid"], plot_df["run_label"])
plt.xlabel("Estimated forward compute (GFLOPs)")
plt.ylabel("FID-1K ↓")
plt.title("DiT compute-quality bubble plot\nBubble size = parameter count")
plt.grid(True, alpha=0.3)
savefig("extra_bubble_gflops_vs_fid.png")

# -----------------------------
# Plot 5: Relative improvement over DiT-Tiny/8 baseline
# -----------------------------
baseline = plot_df[plot_df["run_label"] == "DiT-Tiny-8"]

if len(baseline) == 1:
    base_fid = float(baseline["fid"].iloc[0])
    base_loss = float(baseline["val_loss"].iloc[0])

    rel = plot_df.copy()
    rel["fid_improvement_pct"] = 100.0 * (base_fid - rel["fid"]) / base_fid
    rel["val_loss_improvement_pct"] = 100.0 * (base_loss - rel["val_loss"]) / base_loss

    x = np.arange(len(rel))
    width = 0.35

    plt.figure(figsize=(7, 4))
    plt.bar(x - width / 2, rel["fid_improvement_pct"], width, label="FID improvement")
    plt.bar(x + width / 2, rel["val_loss_improvement_pct"], width, label="Val loss improvement")
    plt.xticks(x, rel["run_label"], rotation=20)
    plt.ylabel("Improvement over DiT-Tiny/8 (%)")
    plt.title("Relative improvement over lowest-compute baseline")
    plt.legend()
    plt.grid(True, axis="y", alpha=0.3)
    savefig("extra_relative_improvement_over_tiny8.png")

    rel_out = RESULT_DIR / "relative_improvement_over_tiny8.csv"
    rel[["run_label", "fid_improvement_pct", "val_loss_improvement_pct"]].to_csv(rel_out, index=False)
    print("Saved:", rel_out)
    display(rel[["run_label", "fid_improvement_pct", "val_loss_improvement_pct"]])
else:
    print("Skipping relative improvement plot: DiT-Tiny-8 baseline not found.")

# -----------------------------
# Plot 6: Timestep-binned validation loss
# -----------------------------
bin_cols = [
    "t_000_100_loss",
    "t_100_300_loss",
    "t_300_600_loss",
    "t_600_1000_loss",
]
available_bin_cols = [c for c in bin_cols if c in plot_df.columns]

if len(available_bin_cols) == 4:
    bin_labels = ["0-100", "100-300", "300-600", "600-1000"]
    x = np.arange(len(bin_labels))

    plt.figure(figsize=(7, 4))
    for _, row in plot_df.iterrows():
        y = [row[c] for c in available_bin_cols]
        plt.plot(x, y, marker="o", label=row["run_label"])

    plt.xticks(x, bin_labels)
    plt.xlabel("Diffusion timestep bin")
    plt.ylabel("Validation denoising loss ↓")
    plt.title("Timestep-binned validation loss")
    plt.legend(fontsize=8)
    plt.grid(True, alpha=0.3)
    savefig("extra_timestep_binned_val_loss.png")
else:
    print("Skipping timestep-binned plot: required timestep loss columns not found.")

print("Done. Extra plots are saved in:", FIG_DIR.resolve())